# 5. End-to-end replay and company adaptation
The end-to-end entry point is `develop("configs/config.yaml")`. It generates or
reuses the fixture, adapts, validates, fits causal features and detectors, tunes
incident persistence, and saves validation evidence plus a frozen model.
This notebook verifies a saved replay without opening the final period.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
# Resolve relative configured output paths consistently from any notebook.
import os

os.chdir(ROOT)
from optical_anomaly.pipeline import prepare, develop, final_evaluation

CONFIG_PATH = ROOT / "configs/config.yaml"
RUN = prepare(CONFIG_PATH)
settings = json.loads((RUN / "settings.json").read_text())
start = pd.Timestamp("2025-01-01", tz="UTC")
boundaries = [
    start + pd.Timedelta(days=settings["generator"]["days"] * f)
    for f in settings["splits"]
]

In [ ]:
import joblib
from optical_anomaly.adapter import TelemetryAdapter
from optical_anomaly.validation import DataValidator
from optical_anomaly.pipeline import score_detectors, incidents_for

if not (RUN / "model.joblib").exists():
    develop(CONFIG_PATH)
model = joblib.load(RUN / "model.joblib")  # Only load trusted local files.
native = pd.read_parquet(
    RUN / "telemetry.parquet",
    columns=["time", "device", "rx_dbm"],
    filters=[("time", "<", boundaries[2])],
)
telemetry = DataValidator(f"{model['interval']}min").transform(
    TelemetryAdapter().transform(native)
)
features = model["engineer"].transform(telemetry)
scores = score_detectors(features, model["statistical"], model["forest"])
validation = scores.loc[model["split"].masks(scores.timestamp)["validation"]]
saved = pd.read_parquet(RUN / "validation_scores.parquet")
pd.testing.assert_frame_equal(validation.reset_index(drop=True), saved)
display(incidents_for(validation, model["policy"], model["interval"]).head())
print("Saved-model replay matches validation scores exactly.")

For company data, configure `TelemetryAdapter(timestamp_column=..., entity_column=...,
metrics={"your_rx_column": "rx_power_dbm"}, units={"rx_power_dbm": "dBm"}, timezone=...)`.
Fit the feature reference and Isolation Forest on a reviewed local baseline;
calibrate scores on a later segment. Reuse the six stage classes directly.
Do not call the synthetic orchestration entry point to fit operator data.

`IncidentManager` retains per-entity state between consecutive calls and can be
saved with joblib. Its output contains closures and active snapshots: upsert using
`incident_id`. Feature extraction currently uses causal batch replay with historical
context, not a bounded-memory streaming feature store. Scheduling, secure serving,
operator calibration and shadow validation remain production work.

## Optional stress check with a fixed incident policy
Refit local references on new data, but retain the default policy when reporting
stress results. Do not pick the winning policy independently for each scenario.

In [ ]:
RUN_STRESS = False
if RUN_STRESS:
    import copy
    import yaml

    rows = []
    scenarios = {
        "new_seed": {"seed": 43},
        "noisy_missing": {"seed": 44, "noise_db": 0.16, "missing_probability": 0.1},
        "healthy": {"seed": 45, "faults_per_entity": 0},
    }
    for name, changes in scenarios.items():
        scenario = copy.deepcopy(settings)
        scenario["generator"].update(entities=12, **changes)
        scenario["output"] = str(ROOT / "outputs" / "stress_demo" / name)
        config_path = ROOT / "outputs" / f"stress_{name}.yaml"
        config_path.write_text(yaml.safe_dump(scenario))
        stress_run = develop(config_path)
        table = pd.read_csv(stress_run / "validation_comparison.csv")
        selected = pd.Series(True, index=table.index)
        for key, value in model["policy"].items():
            selected &= table[key].eq(value)
        rows.append({"scenario": name, **table.loc[selected].iloc[0].to_dict()})
    display(pd.DataFrame(rows))